# 02 · Automatic differentiation and updates

**Goals:** use the chain rule with `backward`, inspect gradients, observe accumulation, and update a parameter. Autograd records operations involving tensors with `requires_grad=True` in a computation graph. Only leaf tensors retain `.grad` by default.

In [ ]:
import torch
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()
print("y =", y.item(), "dy/dx =", x.grad.item())  # 2x + 3 = 7


Gradients accumulate: another backward pass adds to the existing gradient. Build a fresh graph for the second calculation, then clear the gradient.

In [ ]:
(x**2 + 3 * x).backward()
print("after second backward:", x.grad.item())
x.grad.zero_()
print("after clearing:", x.grad.item())


## A manual gradient descent step

For loss `(w - 3)^2`, the derivative is `2(w - 3)`. Subtract the learning rate times the gradient. Use `no_grad` for the update so the update is not part of the next graph.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
for step in range(5):
    loss = (w - 3)**2
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
    w.grad.zero_()
    print(f"step {step + 1}: w={w.item():.3f}, pre-update loss={loss.item():.3f}")


A non-scalar output needs a supplied gradient (a vector-Jacobian product) or reduction to a scalar. Gradients tell us local sensitivity, not an exact loss change for a large step. `detach()` creates a tensor disconnected from the current graph; it may share storage.

In [ ]:
a = torch.tensor([1., 2., 3.], requires_grad=True)
loss = (a**2).mean()
loss.backward()
print("gradient of mean square:", a.grad)
print("detached requires grad:", loss.detach().requires_grad)


## Try it

1. Derive and check the gradient of `x**3 - 2*x` at `x=2`.

2. Remove gradient clearing from the manual loop; compare the path with the original.

3. Try learning rates 0.01, 0.5, and 1.1. Explain the behavior using the derivative.